# Preparación de Datos — Terminal de Transporte

**Fase:** Modelo predictivo — preparación de datos (continuación de `01-exploracion-datos.ipynb`)

Este notebook parte del conjunto de datos completo ya descargado (`data/raw/datos_completos.csv`, generado por el notebook de exploración; no se vuelve a llamar la API) y aplica las decisiones de limpieza respaldadas por la evidencia de la exploración. Cada eliminación o transformación se justifica; nada se quita "porque sí", y **eliminar filas se usa solo como último recurso**.

**Principio metodológico que ordena este notebook:** las transformaciones que *aprenden* información de los datos (por ejemplo, calcular una mediana para reemplazar valores) deben ajustarse **únicamente con los datos de entrenamiento**; si no, información del conjunto de prueba se filtra al proceso (fuga de información). Por eso el notebook tiene dos bloques:

1. **Limpieza determinista** (no aprende nada de los datos, así que puede hacerse antes de partir): derivar variables, eliminar columnas constantes, normalizar textos, completar categorías faltantes.
2. **Separación entrenamiento/prueba** y, después de ella, **el tratamiento de duraciones inválidas**, que sí calcula estadísticas (medianas por ruta) y por lo tanto usa solo el entrenamiento.

**Salida:** `data/processed/train.csv` y `data/processed/test.csv`, listos para modelar.


## Configuración e imports

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

SEED = 42
TEST_SIZE = 0.2
ROUTE_MIN_N = 30     # minimo de viajes plausibles para confiar en la mediana de una ruta
FENCE_H = 12         # horas por encima de la mediana de su ruta a partir de las cuales una duracion se considera error

RAW_PATH = "../data/raw/datos_completos.csv"
TRAIN_PATH = "../data/processed/train.csv"
TEST_PATH = "../data/processed/test.csv"


# Bloque 1 — Limpieza determinista (antes de la separación)

## 1. Cargar los datos y reconstruir tipos y variables derivadas

Se aplican el mismo reconocimiento de tipos y las mismas fórmulas del notebook de exploración (Paso 6): como `fecha_salida`/`fecha_llegada` están truncadas a medianoche en cerca de dos tercios de las filas, la hora de salida real se toma de `fecha_hora_salida_origen` con respaldo en `fecha_salida`, y la de llegada de `hora_de_llegada`. Además de `duracion_viaje_horas`, `hora_salida_decimal` y `dia_semana`, se deriva `mes` (estacionalidad).


In [2]:
df = pd.read_csv(RAW_PATH, dtype=str)
n_original = len(df)
print("Filas cargadas:", f"{n_original:,}", "| columnas:", df.shape[1])

nulos_originales = df.isna().sum()          # se guarda antes de derivar nada

DATE_FMT = "%Y-%m-%dT%H:%M:%S.%f"
date_cols = ["fecha_salida", "fecha_hora_salida_origen", "fecha_llegada", "hora_de_llegada"]
for c in date_cols:
    df[c] = pd.to_datetime(df[c], format=DATE_FMT, errors="coerce")
df["pasajeros"] = pd.to_numeric(df["pasajeros"], errors="coerce")

salida_real = df["fecha_hora_salida_origen"].fillna(df["fecha_salida"])
df["duracion_viaje_horas"] = (df["hora_de_llegada"] - salida_real).dt.total_seconds() / 3600
df["hora_salida_decimal"] = salida_real.dt.hour + salida_real.dt.minute / 60
df["dia_semana"] = salida_real.dt.dayofweek       # 0 = lunes
df["mes"] = salida_real.dt.month
df[["duracion_viaje_horas", "hora_salida_decimal", "dia_semana", "mes"]].describe().T


Filas cargadas: 2,617,130 | columnas: 12


,count,mean,std,min,25%,50%,75%,max
duracion_viaje_horas,2617125.0,5.237335,496.432176,-52620.533333,1.283333,1.983333,3.966667,192858.300000
hora_salida_decimal,2617125.0,12.138547,4.791913,0.000000,8.000000,12.000000,16.000000,23.983333
dia_semana,2617125.0,3.018059,2.018843,0.000000,1.000000,3.000000,5.000000,6.000000
mes,2617125.0,6.730078,3.388312,1.000000,4.000000,7.000000,10.000000,12.000000


## 2. Valores faltantes: identificación, impacto y estrategia

Primero se cuantifican los nulos de las columnas originales. Luego, para cada columna con nulos, se analiza su impacto y se elige una estrategia. Ninguna de las estrategias siguientes requiere calcular estadísticas a partir de los datos, así que pueden aplicarse antes de partir en entrenamiento y prueba.


In [3]:
tabla_nulos = pd.DataFrame({"nulos": nulos_originales, "pct": (nulos_originales / n_original * 100).round(4)})
tabla_nulos[tabla_nulos["nulos"] > 0].sort_values("nulos", ascending=False)


,nulos,pct
fecha_hora_salida_origen,875015,33.4341
empresa,29,0.0011
clase_veh_culo,7,0.0003
fecha_salida,5,0.0002


| Columna | Impacto | Estrategia aplicada | Por qué no se eliminan filas |
|---|---|---|---|
| `fecha_hora_salida_origen` (≈ un tercio de las filas) | Sus nulos no son aleatorios: faltan en el 100% de los registros de febrero a diciembre de 2024 (y en ~71% de enero de 2024) y en ninguno de 2023 ni 2025, es decir, dependen del período (mecanismo MAR, ver exploración, Paso 5). Es la fuente de la hora de salida real cuando `fecha_salida` viene truncada | **Tratamiento específico según el significado de la variable:** completar con `fecha_salida` (respaldo), que justamente trae la hora completa en los períodos en que la otra falta. No es una imputación estadística: se sustituye por la otra fuente que ya tiene el dato real | Eliminar un tercio del dataset (y precisamente un período completo) sesgaría el modelo |
| `empresa`, `clase_veh_culo` (unas decenas de filas en total) | Fracción despreciable, sin patrón | Categoría explícita `DESCONOCIDA` | Se conserva la fila y se le deja al modelo la información de que el dato no existía |
| `fecha_salida` (unas pocas filas) | Cuando falta, `fecha_hora_salida_origen` casi siempre está presente y aporta la hora de salida | Ya cubierto por la estrategia de la fila anterior (`salida_real`) | Solo si faltan ambas fuentes la fila es irrecuperable (última fila de esta tabla) |
| Filas sin **ninguna** fuente de hora de salida | Sin `fecha_hora_salida_origen` ni `fecha_salida` no hay forma de derivar hora, día, mes ni duración | Único caso donde se elimina, por ser irrecuperable | Es la última opción y solo afecta a un puñado de filas |


In [4]:
sin_salida = salida_real.isna()
print(f"Filas sin ninguna hora de salida (irrecuperables): {int(sin_salida.sum())} "
      f"({sin_salida.mean() * 100:.5f}% del total)")
df = df[~sin_salida].copy()

# Fechas de salida imposibles (por ejemplo 2002 o 2029): el anio de salida deberia ser el de llegada
# o el anterior. Cuando no lo es, el calendario de salida no es confiable pero el de llegada si, asi que
# se toman mes y dia de la semana de la llegada en vez de eliminar la fila. (Los viajes que salen el
# 31-12-2022 y llegan en 2023 son legitimos y no se tocan.)
anio_salida = (df["fecha_hora_salida_origen"].fillna(df["fecha_salida"])).dt.year
incompatible = ~(df["hora_de_llegada"].dt.year - anio_salida).between(0, 1)
print(f"Filas con fecha de salida incompatible con la de llegada: {int(incompatible.sum())}")
df.loc[incompatible, "mes"] = df.loc[incompatible, "hora_de_llegada"].dt.month
df.loc[incompatible, "dia_semana"] = df.loc[incompatible, "hora_de_llegada"].dt.dayofweek
df[["dia_semana", "mes"]] = df[["dia_semana", "mes"]].astype(int)
print("Filas restantes:", f"{len(df):,}")


Filas sin ninguna hora de salida (irrecuperables): 5 (0.00019% del total)


Filas con fecha de salida incompatible con la de llegada: 34
Filas restantes: 2,617,125


### Filas exactamente duplicadas

La exploración encontró un pequeño número de filas idénticas en todas sus columnas. **Se conservan**: el recurso no tiene un identificador de viaje que permita confirmar que sean errores de registro (dos viajes de la misma empresa y ruta, con el mismo horario y el mismo número de pasajeros, son posibles), y eliminar filas es el último recurso. Como pueden caer una en entrenamiento y otra en prueba, su efecto sobre la métrica se considera despreciable por su tamaño, y se cuantifica aquí.


In [5]:
n_dup = int(df.duplicated().sum())
print(f"Filas duplicadas: {n_dup:,} ({n_dup / len(df) * 100:.3f}% del total)")


Filas duplicadas: 3,790 (0.145% del total)


## 3. Eliminar columnas de varianza cero

`estado` y `ruta_destino` se confirmaron constantes en la exploración (Paso 4): `estado` solo toma el valor "LLEGADA" (con variantes de mayúsculas) y `ruta_destino` es "MEDELLIN" en el 100% de las filas. Como este recurso contiene únicamente llegadas a Medellín, ninguna de las dos puede aportar nada al modelo: se eliminan.


In [6]:
print("estado:", df["estado"].str.upper().unique().tolist())
print("ruta_destino:", df["ruta_destino"].str.upper().unique().tolist())
df = df.drop(columns=["estado", "ruta_destino"])


estado: ['LLEGADA']


ruta_destino: ['MEDELLIN']


La exploración mostró que `nombre_sucursal` y `subregi_n` aparecen duplicadas por diferencias de mayúsculas/minúsculas (dos terminales reales aparecen como cuatro valores; cinco subregiones como diez), consecuencia de los dos sistemas de captura. Se normaliza todo a mayúsculas sin espacios sobrantes y los nulos se marcan con la categoría explícita `DESCONOCIDA`.

**`ruta_origen` tiene además un problema de formato:** el mismo origen se registra unas veces solo (`RIONEGRO`) y otras con el destino pegado (`RIONEGRO-MEDELLIN`). Como `ruta_destino` es siempre MEDELLIN, ese sufijo no aporta información: es el mismo origen escrito de dos formas. Dejarlo partiría cada ruta en dos categorías (casi 200 orígenes aparecen en ambas formas, con duraciones prácticamente idénticas), duplicaría el aprendizaje con la mitad de datos en cada una y debilitaría las medianas por ruta del Paso 7. Se elimina el sufijo `-MEDELLIN` (tolerando espacios alrededor del guion, como en `MOMPOX - MEDELLIN`). Es una limpieza de texto que no aprende nada de los datos, así que puede hacerse antes de la separación sin riesgo de fuga.

In [7]:
categorical_cols = ["nombre_sucursal", "empresa", "ruta_origen", "subregi_n", "clase_veh_culo"]

print("Cardinalidad ANTES de normalizar:")
print({c: int(df[c].nunique()) for c in categorical_cols})

for c in categorical_cols:
    df[c] = df[c].fillna("DESCONOCIDA").astype(str).str.upper().str.strip()

# ruta_origen: quitar el destino pegado ("RIONEGRO-MEDELLIN" -> "RIONEGRO")
SUFIJO_DESTINO = r"\s*-\s*MEDELLIN$"
con_sufijo = df["ruta_origen"].str.contains(SUFIJO_DESTINO, regex=True)
print(f"\nFilas de ruta_origen con sufijo -MEDELLIN: {int(con_sufijo.sum()):,} ({con_sufijo.mean() * 100:.1f}%)")
df["ruta_origen"] = df["ruta_origen"].str.replace(SUFIJO_DESTINO, "", regex=True).str.strip()
assert (df["ruta_origen"] != "").all()

for c in categorical_cols:
    df[c] = df[c].astype("category")

print("\nCardinalidad DESPUES de normalizar:")
print({c: int(df[c].nunique()) for c in categorical_cols})
print()
print(df["nombre_sucursal"].value_counts())
print(df["subregi_n"].value_counts())


Cardinalidad ANTES de normalizar:


{'nombre_sucursal': 4, 'empresa': 169, 'ruta_origen': 412, 'subregi_n': 10, 'clase_veh_culo': 6}



Filas de ruta_origen con sufijo -MEDELLIN: 915,512 (35.0%)



Cardinalidad DESPUES de normalizar:
{'nombre_sucursal': 2, 'empresa': 167, 'ruta_origen': 212, 'subregi_n': 5, 'clase_veh_culo': 6}

nombre_sucursal
TERMINAL DEL NORTE    1912009
TERMINAL DEL SUR       705116
Name: count, dtype: int64
subregi_n
ORIENTE      1025181
NORTE         773882
SUR           411902
OCCIDENTE     234432
CENTRO        171728
Name: count, dtype: int64


## 5. Eliminar las columnas de fecha originales

De ellas ya se extrajo todo lo necesario (`duracion_viaje_horas`, `hora_salida_decimal`, `dia_semana`, `mes`). Se eliminan las cuatro, cada una por una razón distinta:

| Columna | Por qué se elimina |
|---|---|
| `fecha_llegada` | Fuga de información: es una de las columnas con que se calcula el objetivo, y en un caso real no se conoce al momento de predecir |
| `hora_de_llegada` | Fuga de información: es la llegada real con que se calcula `duracion_viaje_horas` |
| `fecha_hora_salida_origen` | No es fuga (se conoce al salir), pero su información útil ya quedó en `hora_salida_decimal`, `dia_semana` y `mes`; mantenerla aparte reintroduciría sus nulos sin aportar nada nuevo |
| `fecha_salida` | No es fuga, pero ya se extrajo todo lo útil de ella |


In [8]:
df = df.drop(columns=date_cols)
print("Columnas restantes:", list(df.columns))


Columnas restantes: ['nombre_sucursal', 'empresa', 'ruta_origen', 'subregi_n', 'clase_veh_culo', 'pasajeros', 'duracion_viaje_horas', 'hora_salida_decimal', 'dia_semana', 'mes']


# Bloque 2 — Separación entrenamiento/prueba y tratamiento del objetivo

## 6. Separación entrenamiento/prueba

- **Porcentajes:** 80% entrenamiento, 20% prueba (con más de 2 millones de filas, el 20% de prueba son cientos de miles de viajes: de sobra para una evaluación confiable).
- **Método:** separación **aleatoria** con semilla fija (`random_state=42`, reproducible).
- **Por qué aleatoria y no cronológica:** el problema no es de series de tiempo. Se predice la duración de *un viaje individual* a partir de sus propias características (ruta, empresa, hora, día, mes...), tratando cada viaje como un caso independiente; el modelo no usa valores pasados de la serie. Además, los datos cubren tres años completos, así que la separación aleatoria mezcla todos los meses y años en ambos conjuntos.
- **Grupos naturales:** en este dominio los mismos servicios (empresa + ruta + horario) se repiten a diario. Eso significa que viajes casi idénticos caerán tanto en entrenamiento como en prueba; sin embargo, en uso real el modelo también se aplicará a servicios que ya existen, así que la separación aleatoria refleja el escenario de uso. Aun así, es una fuente de optimismo en la métrica, que se contrastará en la evaluación con una validación por grupos (empresa/ruta).


In [9]:
train, test = train_test_split(df, test_size=TEST_SIZE, random_state=SEED)
print(f"Entrenamiento: {len(train):,} filas ({len(train)/len(df)*100:.0f}%)")
print(f"Prueba:        {len(test):,} filas ({len(test)/len(df)*100:.0f}%)")

print("\nProporcion por clase de vehiculo (entrenamiento vs prueba):")
pd.DataFrame({"train %": train["clase_veh_culo"].value_counts(normalize=True) * 100,
              "test %": test["clase_veh_culo"].value_counts(normalize=True) * 100}).round(2)


Entrenamiento: 2,093,700 filas (80%)
Prueba:        523,425 filas (20%)

Proporcion por clase de vehiculo (entrenamiento vs prueba):


,train %,test %
clase_veh_culo,,
BUS,50.62,50.35
MICROBUS,27.36,27.44
CAMIONETA,20.06,20.25
AUTOMOVIL,1.54,1.53
DUO BUS,0.43,0.42
DESCONOCIDA,0.00,0.00


## 7. Duraciones inválidas: reemplazar, no eliminar

**Qué se encontró en la exploración (Paso 7):** hay duraciones negativas (imposibles) y otras de cientos de horas, casi seguro errores de captura en la fecha (desfases de días enteros). Pero **no todo lo que supera 24 horas es un error**: hay rutas genuinamente largas (por ejemplo Maicao o Mocoa) cuyos viajes reales duran entre 24 y 36 horas. Un corte fijo destruiría esos viajes, y reemplazarlos por una mediana global (~2 horas) sería aún peor.

**Regla aplicada.** Una duración se considera inválida si:

- es **negativa**, o
- supera en más de **12 horas** la duración típica (mediana) **de su propia ruta**.

El umbral de 12 horas no toca los retrasos reales (un viaje demorado sigue siendo información valiosa que el modelo debe aprender), pero sí captura los desfases de un día completo (±24 h), que son la firma de los errores observados.

**Reemplazo.** Los valores inválidos se sustituyen por la **mediana de su ruta** (o por la mediana global si la ruta tiene menos de 30 viajes plausibles). Es la misma idea de reemplazo por mediana (más robusta que la media frente a valores extremos) pero calculada dentro de cada ruta, porque la exploración mostró que la duración depende fuertemente de la ruta.

**Sin fuga de información:** las medianas por ruta se calculan **solo con el conjunto de entrenamiento** y luego se aplican tanto a entrenamiento como a prueba. Se agrega la columna `duracion_corregida` (verdadero/falso) para saber qué filas fueron corregidas; esa columna **no es una variable predictora** (se deriva del objetivo) y solo sirve para reportar métricas sobre datos no corregidos.


In [10]:
plausible = train["duracion_viaje_horas"].between(0, 72)
tab = (train[plausible].groupby("ruta_origen", observed=True)["duracion_viaje_horas"]
       .agg(["median", "count"]))
med_ruta = {str(k): v for k, v in tab["median"].where(tab["count"] >= ROUTE_MIN_N).dropna().items()}
med_global = train.loc[plausible, "duracion_viaje_horas"].median()
print(f"Rutas con mediana propia (>= {ROUTE_MIN_N} viajes plausibles): {len(med_ruta)} de {train['ruta_origen'].nunique()}")
print(f"Mediana global (respaldo): {med_global:.2f} h")


def corregir_duraciones(d):
    d = d.copy()
    ref = d["ruta_origen"].astype(str).map(med_ruta).fillna(med_global)
    invalida = (d["duracion_viaje_horas"] < 0) | (d["duracion_viaje_horas"] > ref + FENCE_H)
    d["duracion_original"] = d["duracion_viaje_horas"]
    d["duracion_corregida"] = invalida
    d.loc[invalida, "duracion_viaje_horas"] = ref[invalida]
    return d


train = corregir_duraciones(train)
test = corregir_duraciones(test)

for nombre, d in [("entrenamiento", train), ("prueba", test)]:
    n = int(d["duracion_corregida"].sum())
    print(f"{nombre}: {n:,} duraciones corregidas ({n / len(d) * 100:.3f}%)")


Rutas con mediana propia (>= 30 viajes plausibles): 183 de 210
Mediana global (respaldo): 1.98 h


entrenamiento: 3,074 duraciones corregidas (0.147%)
prueba: 713 duraciones corregidas (0.136%)


In [11]:
print("Distribucion de la duracion (entrenamiento) ANTES de corregir:")
print(train["duracion_original"].describe().round(2))
print("\nDESPUES de corregir:")
print(train["duracion_viaje_horas"].describe().round(2))

conservados = ((train["duracion_original"] > 24) & (train["duracion_original"] <= 36) & ~train["duracion_corregida"]).sum()
print(f"\nViajes de 24 a 36 h que se CONSERVARON intactos (rutas largas reales): {int(conservados):,}")


Distribucion de la duracion (entrenamiento) ANTES de corregir:
count    2093700.00
mean           4.89
std          446.03
min       -52620.53
25%            1.28
50%            1.98
75%            3.97
max       192858.30
Name: duracion_original, dtype: float64

DESPUES de corregir:


count    2093700.00
mean           3.49
std            3.60
min            0.00
25%            1.28
50%            1.98
75%            3.97
max           33.32
Name: duracion_viaje_horas, dtype: float64

Viajes de 24 a 36 h que se CONSERVARON intactos (rutas largas reales): 2,323


In [12]:
train[train["duracion_corregida"]].sample(min(12, int(train["duracion_corregida"].sum())), random_state=SEED)[
    ["ruta_origen", "duracion_original", "duracion_viaje_horas"]].round(2)


,ruta_origen,duracion_original,duracion_viaje_horas
1662476,PASTO,32.82,18.63
1408629,SONSON,17.47,4.10
332917,TURBO,24.45,8.23
596064,PUERTO BERRIO,15.90,3.77
1066369,SOPETRAN,13.73,1.18
1065889,CARTAGENA,26.78,14.72
1379169,CUCUTA,43.12,15.60
1262706,SAN FRANCISCO,23.60,2.82
1194953,BUCARAMANGA,33.55,7.98
2236969,SAN PEDRO DE LOS MILAGROS,13.42,1.27


**Lectura de la corrección:**

- Se corrigieron **3.074 duraciones en entrenamiento (0.147%)** y **713 en prueba (0.136%)**: proporciones casi idénticas en ambos conjuntos, como es de esperar en una separación aleatoria.
- La corrección **no altera el centro de la distribución** (mediana de 1.98 h y cuartiles de 1.28 h y 3.97 h intactos), pero elimina el efecto de los valores imposibles: la desviación estándar pasa de **446 h a 3.6 h**, la media de 4.89 h a 3.49 h y el rango de [-52.620 h, 192.858 h] a [0 h, 33.3 h].
- Se **conservaron intactos 2.323 viajes de entre 24 y 36 horas** del conjunto de entrenamiento (rutas largas reales) que un corte fijo de 24 h habría destruido.
- La tabla de ejemplos muestra que cada valor imposible se sustituye por la duración típica **de su propia ruta**, no por un valor global de ~2 h: un viaje desde Cúcuta de 43.1 h pasa a 15.6 h, uno desde Pasto de 32.8 h pasa a 18.6 h y uno desde Sopetrán de 13.7 h pasa a 1.2 h.
- **Limitación a tener presente:** las duraciones reemplazadas son estimaciones (la mediana de la ruta), no mediciones. Por eso se conserva la columna `duracion_corregida`: al evaluar el modelo puede reportarse la métrica también solo sobre las filas no corregidas (más del 99.8% de los datos), para comprobar que el resultado no depende de valores reemplazados.


## 8. Categorías de alta cardinalidad: se conservan tal cual

`ruta_origen` y `empresa` tienen muchas categorías, algunas con pocos viajes. **No se agrupan en una categoría "OTROS"**: la duración de un viaje depende de la ruta y de la empresa (la exploración, Paso 9, muestra rangos de duración mediana muy amplios entre ellas), así que agrupar categorías descartaría justamente la señal más útil del problema. Lo que sí hay que prever es que en producción pueden aparecer categorías no vistas en el entrenamiento; la codificación del modelo deberá tolerarlas (por ejemplo `handle_unknown="ignore"`).


In [13]:
for c in ["ruta_origen", "empresa"]:
    vistas = set(train[c].astype(str).unique())
    no_vistas = ~test[c].astype(str).isin(vistas)
    print(f"{c}: {train[c].nunique()} categorias en entrenamiento; "
          f"filas de prueba con categoria no vista: {int(no_vistas.sum()):,} ({no_vistas.mean() * 100:.4f}%)")


ruta_origen: 210 categorias en entrenamiento; filas de prueba con categoria no vista: 4 (0.0008%)


empresa: 162 categorias en entrenamiento; filas de prueba con categoria no vista: 5 (0.0010%)


## 9. Prevención de fuga de información

| Verificación | Cómo se cumple en este notebook |
|---|---|
| La variable objetivo no se usa como predictora | Las predictoras se listan explícitamente abajo. `duracion_corregida` y `duracion_original` se derivan del objetivo y **no** se incluyen |
| El conjunto de prueba no interviene en el entrenamiento | La separación se hace antes de calcular cualquier estadística; las medianas por ruta salen solo del entrenamiento |
| Las transformaciones que aprenden de los datos se ajustan solo con entrenamiento | Medianas por ruta: solo entrenamiento. La codificación de categóricas y cualquier escalado se ajustarán, en el notebook de modelado, dentro de un *pipeline* entrenado solo con entrenamiento |
| No se usan variables conocidas solo después del evento | Se eliminaron `fecha_llegada` y `hora_de_llegada`. Las variables de tiempo (`hora_salida_decimal`, `dia_semana`, `mes`) salen de la hora de **salida** |
| Supuestos a justificar | **`pasajeros`** se registra en el terminal de llegada; se asume que el número de pasajeros es conocido al salir (manifiesto del viaje), aunque podría incluir pasajeros que abordan en ruta. Al ser la variable más correlacionada con la duración, el modelado comparará el desempeño **con y sin** ella. `nombre_sucursal` (terminal de destino) se asume conocida al salir |


## 10. Guardar los conjuntos de entrenamiento y prueba

In [14]:
predictoras = ["nombre_sucursal", "empresa", "ruta_origen", "subregi_n", "clase_veh_culo",
               "pasajeros", "hora_salida_decimal", "dia_semana", "mes"]
objetivo = "duracion_viaje_horas"
columnas_salida = predictoras + [objetivo, "duracion_corregida"]

os.makedirs(os.path.dirname(TRAIN_PATH), exist_ok=True)

train[columnas_salida].to_csv(TRAIN_PATH, index=False)
test[columnas_salida].to_csv(TEST_PATH, index=False)

print(f"train: {len(train):,} filas | test: {len(test):,} filas | columnas: {len(columnas_salida)}")
print(f"Filas conservadas del dataset original: {(len(train) + len(test)) / n_original * 100:.4f}% "
      f"({n_original - len(train) - len(test)} eliminadas)")
print("\nPredictoras:", predictoras)
print("Objetivo:", objetivo)
train[columnas_salida].head()


train: 2,093,700 filas | test: 523,425 filas | columnas: 11
Filas conservadas del dataset original: 99.9998% (5 eliminadas)

Predictoras: ['nombre_sucursal', 'empresa', 'ruta_origen', 'subregi_n', 'clase_veh_culo', 'pasajeros', 'hora_salida_decimal', 'dia_semana', 'mes']
Objetivo: duracion_viaje_horas


,nombre_sucursal,empresa,ruta_origen,subregi_n,clase_veh_culo,pasajeros,hora_salida_decimal,dia_semana,mes,duracion_viaje_horas,duracion_corregida
905055,TERMINAL DEL NORTE,EXPRE - BELMIRA S.A.,SAN PEDRO DE LOS MILAGROS,NORTE,CAMIONETA,8,10.000000,4,2,1.150000,False
2382553,TERMINAL DEL NORTE,COONORTE,PUERTO BERRIO,NORTE,MICROBUS,5,4.750000,2,10,3.600000,False
114852,TERMINAL DEL NORTE,TRANS CISNEROS-ENTRERRIOS,MACEO,NORTE,BUS,12,8.000000,4,2,2.600000,False
1754019,TERMINAL DEL NORTE,TRANS CISNEROS-ENTRERRIOS,ENTRERRIOS,NORTE,CAMIONETA,6,5.500000,0,1,2.083333,False
2325800,TERMINAL DEL NORTE,FLOTA EL CARMEN,EL CARMEN DE VIBORAL,ORIENTE,MICROBUS,19,7.416667,1,9,1.600000,False


# Resumen de lo aplicado en este notebook

Partiendo de los 2.617.130 registros completos:

1. **Variables derivadas** con la fórmula corregida de la exploración: `duracion_viaje_horas`, `hora_salida_decimal`, `dia_semana` y `mes`.
2. **Valores faltantes:** `fecha_hora_salida_origen` (33.43%) se complementó con `fecha_salida` (sin imputación estadística); `empresa` y `clase_veh_culo` faltantes pasaron a la categoría `DESCONOCIDA` (36 filas); solo se eliminaron **5 filas** sin ninguna hora de salida (0.0002%). Se conserva el 99.9998% de los datos.
3. **Fechas de salida imposibles** (34 filas): se conservaron, tomando mes y día de la semana de la fecha de llegada.
4. **Filas duplicadas** (3.790; 0.145%): se conservaron, porque sin un identificador de viaje no hay evidencia de que sean errores.
5. **Columnas eliminadas:** `estado` y `ruta_destino` (varianza cero) y las 4 columnas de fecha originales (`fecha_llegada` y `hora_de_llegada` por fuga de información; `fecha_salida` y `fecha_hora_salida_origen` porque ya se extrajo su información útil).
6. **Textos normalizados:** `nombre_sucursal` de 4 a 2 valores, `subregi_n` de 10 a 5, `clase_veh_culo` ("bus" pasa a "BUS") y `ruta_origen` de 412 a 212 valores (se quitó el sufijo `-MEDELLIN` que el 35.0% de las filas traía pegado al origen, p. ej. `RIONEGRO-MEDELLIN` pasa a `RIONEGRO`).
7. **Separación entrenamiento/prueba** aleatoria 80/20 (semilla 42): 2.093.700 y 523.425 filas.
8. **Duraciones inválidas: reemplazadas, no eliminadas.** Se marcaron con un criterio por ruta (negativas, o más de 12 h sobre la mediana de su ruta) y se sustituyeron por la mediana de su ruta calculada **solo con el entrenamiento** (3.074 filas en entrenamiento y 713 en prueba). La columna `duracion_corregida` registra cuáles fueron y **no es una variable predictora**.
9. **Categorías sin agrupar:** `ruta_origen` (210 categorías en entrenamiento) y `empresa` (162). En prueba solo 4 filas (0.0008%) tienen una `ruta_origen` no vista en entrenamiento y 5 (0.0010%) una `empresa` no vista.
10. **Fuga de información:** revisada explícitamente en la sección 9.

**Salida:** `data/processed/train.csv` y `data/processed/test.csv`, con 9 predictoras, el objetivo y la bandera `duracion_corregida`.

**Pendiente para el modelado:** codificar las categóricas de alta cardinalidad (con tolerancia a categorías no vistas) y escalar donde el algoritmo lo requiera, siempre dentro de un *pipeline* ajustado solo con entrenamiento; construir el modelo de referencia (*baseline*) y el modelo predictivo; justificar la métrica; comparar el desempeño con y sin `pasajeros`; y evaluar también sobre las filas no corregidas.
